# 다국어 EU 법령 분류 임베딩과 전처리

민건우 A 담당의 cache, 전처리, 통계, 인수인계와 first-chunk 검증 코드입니다.

- **기본 실행 `RUN_MODE='repair_existing'`**: 기존 E5 배열은 재인코딩하지 않고 hash·CELEX·전체 norm을 검사합니다. 원문 parquet 24개에서 `len(text.split())`로 단어 수와 fertility를 고치고, 21차원 label·metadata·로더를 추가합니다. 바뀌는 기존 통계와 표는 `_repair_backups`에 백업합니다.
- **새 생성 `RUN_MODE='generate'`**: 별도 `_fp32_v2` 폴더를 사용하며, 기본 계산은 fp32입니다. notebook 01에서 만든 `01_splits/celex_splits.json`을 새 폴더에 먼저 복사하세요. `USE_FP16=True`일 때는 EN validation의 고정 100문서 cosine 검증이 필요합니다.
- **통계 정의**: 본문 token 수와 word 수를 사용합니다. `exceeds_512`는 기존 진단과 같은 본문 + CLS/SEP(2) 기준이며, 본문만/쿼리 포함 비율은 별도 열로 남깁니다.
- **first-chunk LR**: 핵심 실험 확정 뒤 B의 고정 C·threshold를 `cache_metadata/classifier_settings.json`에 넣고 `CORE_RESULTS_FINALIZED=True`, `RUN_FIRST_CHUNK_ROBUSTNESS=True`로 실행하세요. template의 null을 실제 확정값으로 바꿔야 합니다. 실험 결과는 실행 전까지 존재하지 않습니다.
- 기존 cache의 과거 model commit·계산 정밀도·환경은 추정하지 않고 unknown으로 보존합니다. 기존 TF-IDF 행렬도 기본 보정 모드에서는 덮어쓰지 않습니다.

Colab/Jupyter 환경에 numpy, pandas, pyarrow, scipy, scikit-learn, torch, transformers, tqdm이 필요합니다. Drive 파일 다운로드에는 google-api-python-client와 계정 인증이 필요합니다. 아래 셀을 위에서부터 실행하세요.

수정본에는 이전 코드의 저장 출력을 제거했습니다. 로컬 검증 범위와 GPU 전체 실행 여부는 별도 변경 안내에 기록합니다.


In [1]:
# 0. 환경 설정 및 데이터 준비
import os
import io
import re
import time
import json
import math
import shutil
import hashlib
import unicodedata
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

from sklearn.feature_extraction.text import TfidfVectorizer as SklearnTfidfVectorizer

# 통계와 분류기 정의를 환경에 따라 바꾸지 않습니다.
HAS_RAPIDS = False

try:
    from google.colab import drive, auth
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

LANGUAGES = ["en", "de", "fr", "el", "fi", "pl", "bg", "mt"]
TARGET_LANGUAGES = ["de", "fr", "el", "fi", "pl", "bg", "mt"]
SPLITS = ["train", "validation", "test"]

MODEL_NAME = "intfloat/multilingual-e5-small"
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

PARQUET_FILE_IDS: Dict[str, str] = {
    "en_train.parquet": "1MNHTW1PxQ54KJD4Tvkk733GcXV8g4Dsc",
    "en_validation.parquet": "1UJCjLZrgR9uGbYqFYj-fBzIkGeWOB0Xj",
    "en_test.parquet": "1hbzkBPZrl0RT876omnyapdTjY-EedNoy",
    "de_train.parquet": "1THdhSGbsu_Vy-Mw_qcik4hCo5xzjakZ_",
    "de_validation.parquet": "1RIuCbwMprgdrglqIe5ttTVvzzRO2uuGj",
    "de_test.parquet": "1dM8eAHnFTR2QfqsILltoTvfCwciCjRqg",
    "fr_train.parquet": "1w3U-sfe1bnMFpdBBEvipAigpFbeR363o",
    "fr_validation.parquet": "1d3wVvu2ZgrbmaWQjciga4ssFmXEoOPaa",
    "fr_test.parquet": "1ltUkHFZHapTWagX0kf-PVwVlW2CaxYu8",
    "el_train.parquet": "1L7_Hj_uOL6ST8dVpUmVtRceXmIWEy76S",
    "el_validation.parquet": "1HccvR4ohm-5PPCYvSnTjv-PiAoLxM3h5",
    "el_test.parquet": "1YzYTQ1yZzR_Y8nGqEZIUXudTOuB7iPDp",
    "fi_train.parquet": "18H0CGgRBerfHXNfuxyTtA4MQsY2jHhnu",
    "fi_validation.parquet": "1sN9c55wwKlzaNX1QAzSea-4cxMKBdFci",
    "fi_test.parquet": "1sivXdyoN7_Mdd7iEbyU54Fpouinku76U",
    "pl_train.parquet": "1kgG13P6fYI1oQ6nLBKUMQB84_ub-LgEN",
    "pl_validation.parquet": "1yQC9ei6Puw6JQr7VcrmQSKTT9DBJmZGy",
    "pl_test.parquet": "1DoxpHJ2Vk2kn-NDwRcW4J2pOZ4r_XoOb",
    "bg_train.parquet": "1lDGJYblrwCmgIqjGYBrnW7VV4oLIR77c",
    "bg_validation.parquet": "1iIR1JFvLXWuLd46Mlt5rFvUTPWV-HQ5p",
    "bg_test.parquet": "1W871jCFWvoFLHiuxuSKFmS3EH2E2Y4pk",
    "mt_train.parquet": "1-y2CaVngupZ9mBVUubrtoFtovotAlg3l",
    "mt_validation.parquet": "1s0_LUkfEsSin0aU2659E6Pfe0myGUeE3",
    "mt_test.parquet": "1bhjhAS757ZhnWmLPCJuTmp1F6DOhetkT",
}

CANDIDATE_DATA_DIRS = [
    Path("/content/local_data"),
    Path("/content/drive/MyDrive/기계학습/data"),
    Path("/content/drive/.shortcut-targets-by-id/1yHEq5aNdbhY50dqgj963aBcEoQDOY_Pz/data"),
]
DATA_DIR = next(
    (p for p in CANDIDATE_DATA_DIRS if p.exists() and (p / "en_test.parquet").exists()),
    Path("/content/local_data"),
)
DATA_DIR.mkdir(parents=True, exist_ok=True)


def ensure_all_24_parquet_files(target_dir: Path = DATA_DIR):
    missing = [fname for fname in PARQUET_FILE_IDS if not (target_dir / fname).exists()]
    if not missing:
        print(f"All 24 parquet files ready: {target_dir}")
        return
    print(f"Downloading {len(missing)} parquet files to {target_dir}...")
    if IN_COLAB:
        auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    service = build("drive", "v3")
    for fname in tqdm(missing, desc="Downloading parquet files"):
        file_id = PARQUET_FILE_IDS[fname]
        out_path = target_dir / fname
        request = service.files().get_media(fileId=file_id)
        with open(out_path, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request, chunksize=32 * 1024 * 1024)
            done = False
            while not done:
                _, done = downloader.next_chunk()
    print(f"Download complete: {target_dir}")


ensure_all_24_parquet_files(DATA_DIR)

# 기존 캐시 통계/인수인계만 보완하는 것이 기본값입니다.
from datetime import datetime, timezone
RUN_MODE = 'repair_existing'       # 'generate': 별도 새 폴더에서 E5 새 생성
USE_FP16 = False                    # True면 고정 100문서 검증 통과 후에만 허용
ENCODE_BATCH_SIZE = 256
RUN_TFIDF = RUN_MODE == 'generate'
RUN_ALIGNMENT = True
RUN_P3_MAXPOOL = False
RUN_P3_MRR = False
RUN_FIRST_CHUNK_ROBUSTNESS = False
CORE_RESULTS_FINALIZED = False
CREATE_ZIP = False
MODEL_REVISION = None               # 새 생성 시 실제 resolved commit을 기록/고정

if RUN_MODE not in ['repair_existing','generate']:
    raise ValueError('RUN_MODE must be repair_existing or generate')
EXISTING_WORK_DIR = Path('/content/drive/MyDrive/기계학습_산출물') if IN_COLAB else Path('./outputs')
WORK_DIR = EXISTING_WORK_DIR if RUN_MODE == 'repair_existing' else EXISTING_WORK_DIR.with_name(EXISTING_WORK_DIR.name+'_fp32_v2')
if RUN_MODE == 'repair_existing' and not WORK_DIR.exists():
    raise FileNotFoundError(f'기존 산출물 폴더를 확인하세요: {WORK_DIR}')
DIRS = {
    'splits':WORK_DIR/'01_splits', 'stats':WORK_DIR/'02_tokenization_stats',
    'emb_core':WORK_DIR/'02_embeddings_core_480_mean', 'emb_p2_first':WORK_DIR/'02_embeddings_p2_first_chunk',
    'emb_p3':WORK_DIR/'02_embeddings_p3_extensions', 'labels':WORK_DIR/'03_labels',
    'tfidf_rq1':WORK_DIR/'04_tfidf_rq1', 'rq4':WORK_DIR/'07_rq4_alignment',
    'meta':WORK_DIR/'cache_metadata', 'robustness':WORK_DIR/'08_first_chunk_robustness',
}
for folder in DIRS.values():
    folder.mkdir(parents=True,exist_ok=True)
REPAIR_RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPAIR_BACKUP_DIR = WORK_DIR/'_repair_backups'/REPAIR_RUN_ID
CLASSIFIER_SETTINGS_PATH = DIRS['meta']/'classifier_settings.json'
print(f'Mode={RUN_MODE}, output={WORK_DIR}, fp16={USE_FP16}')


Mounted at /content/drive


Download complete: /content/local_data
Mode=repair_existing, output=/content/drive/MyDrive/기계학습_산출물, fp16=False


In [2]:
# 1. CELEX, 라벨, 통계 검증
import itertools
import platform
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

SPLIT_SIZES = {'main_train': 10000, 'adaptation_pool': 4604, 'validation': 5000, 'test': 5000}
RAW_SPLITS = {'main_train': 'train', 'adaptation_pool': 'train', 'validation': 'validation', 'test': 'test'}
LABEL_INDICES = list(range(21))
CODE_SCHEMA_VERSION = 2
EXPECTED_TEST_STATS = {
    'en': (5.75, 2, 574), 'de': (6.42, 2, 630), 'fr': (7.02, 2, 715),
    'el': (7.81, 3, 807), 'fi': (6.64, 2, 679), 'pl': (6.75, 2, 694),
    'bg': (6.48, 2, 648), 'mt': (10.91, 3, 1142),
}

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)

def save_npy_atomic(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'wb') as f:
        np.save(f, value, allow_pickle=False)
    os.replace(tmp, path)

def backup_before_repair(path):
    path = Path(path)
    if RUN_MODE == 'repair_existing' and path.exists():
        dest = REPAIR_BACKUP_DIR / path.relative_to(WORK_DIR)
        if not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, dest)

def detect_columns(cols):
    groups = [('celex_id', 'id', 'doc_id', 'celex'), ('text', 'document', 'body', 'content'),
              ('labels', 'eurovoc_concepts', 'level_1_labels', 'label')]
    result = [next((name for name in group if name in cols), None) for group in groups]
    if None in result:
        raise ValueError(f'CELEX/text/label 열을 확인하세요: {list(cols)}')
    return tuple(result)

def load_aligned_raw(lang, raw_split, ids, data_dir=DATA_DIR):
    # 모든 실행 환경에서 같은 Python whitespace 정의를 사용합니다.
    df = pd.read_parquet(Path(data_dir) / f'{lang}_{raw_split}.parquet')
    id_col, text_col, label_col = detect_columns(df.columns)
    df = df[[id_col, text_col, label_col]].rename(columns={id_col:'celex_id', text_col:'text', label_col:'labels'})
    df['celex_id'] = df['celex_id'].astype(str)
    if not df.celex_id.is_unique:
        raise ValueError(f'{lang}/{raw_split}: 중복 CELEX')
    df = df.set_index('celex_id').loc[list(ids)]
    if df.text.isna().any() or df.labels.isna().any():
        raise ValueError(f'{lang}/{raw_split}: text 또는 labels 결측')
    return df

def labels_to_multihot(label_series):
    y = np.zeros((len(label_series), 21), dtype=np.int8)
    for i, labels in enumerate(label_series):
        values = list(labels)
        if any(not isinstance(v, (int, np.integer)) or not 0 <= int(v) < 21 for v in values):
            raise ValueError('Level 1 labels는 원본의 정수 index 0~20이어야 합니다.')
        y[i, values] = 1
    return y

def load_lang_split_texts_and_word_counts(lang, split, ordered_celex_list, data_dir=DATA_DIR):
    raw = load_aligned_raw(lang, split, ordered_celex_list, data_dir)
    texts = raw.text.tolist()
    return texts, np.array([len(t.split()) for t in texts], dtype=np.int32)

def validate_split_ids(split_info, expected_sizes=SPLIT_SIZES):
    for split, size in expected_sizes.items():
        ids = split_info[split]
        if len(ids) != size or len(set(ids)) != size or ids != sorted(ids):
            raise ValueError(f'{split}: 크기/중복/CELEX 정렬 오류')
    for a, b in itertools.combinations(expected_sizes, 2):
        if set(split_info[a]) & set(split_info[b]):
            raise ValueError(f'CELEX leakage: {a} / {b}')
    common = sorted(split_info['main_train'] + split_info['adaptation_pool'])
    selected = np.random.default_rng(SEED).choice(len(common), size=expected_sizes['main_train'], replace=False)
    if sorted(common[i] for i in selected) != split_info['main_train']:
        raise ValueError('seed=42 main train 추출과 저장 목록이 다릅니다.')

def validate_embedding(path, ids):
    arr = np.load(path, mmap_mode='r', allow_pickle=False)
    if arr.shape != (len(ids), 384) or arr.dtype != np.float32:
        raise ValueError(f'Embedding shape/dtype 오류: {path}')
    for start in range(0, len(arr), 1024):
        batch = arr[start:start+1024]
        if not np.isfinite(batch).all() or not np.allclose(np.linalg.norm(batch, axis=1), 1, atol=1e-5):
            raise ValueError(f'Embedding NaN/Inf/L2 norm 오류: {path}')
    return arr

def repair_document_statistics(stats, texts, ids, lang, split):
    stats = stats.copy()
    if stats.celex_id.tolist() != list(ids) or len(texts) != len(ids):
        raise ValueError(f'{lang}/{split}: 원문과 통계 CELEX 순서 불일치')
    stats['num_words'] = np.array([len(t.split()) for t in texts], dtype=np.int32)
    stats['fertility'] = np.divide(stats.num_tokens, stats.num_words,
                                   out=np.full(len(stats), np.nan), where=stats.num_words.to_numpy() > 0)
    # 기존 tokenizer_diagnostics.ipynb는 query 없이 본문 + CLS/SEP를 측정합니다.
    stats['exceeds_512'] = (stats.num_tokens + 2) > 512
    stats['exceeds_512_content_only'] = stats.num_tokens > 512
    stats['exceeds_512_with_query'] = (stats.num_tokens + 5) > 512
    stats['language'] = lang
    stats['split'] = split
    expected_chunks = np.maximum(1, (stats.num_tokens.to_numpy() + 479) // 480)
    if not np.array_equal(stats.num_chunks.to_numpy(), expected_chunks):
        raise ValueError(f'{lang}/{split}: 480-token chunk 수 불일치')
    counts = stats[['num_words','num_tokens','num_chunks','num_unk']]
    if counts.isna().any().any() or (counts < 0).any().any() or (stats.num_unk > stats.num_tokens).any():
        raise ValueError(f'{lang}/{split}: 유효하지 않은 통계')
    if split == 'test':
        mean, median, maximum = EXPECTED_TEST_STATS[lang]
        actual = (round(float(stats.num_chunks.mean()), 2), float(stats.num_chunks.median()), int(stats.num_chunks.max()))
        if actual != (mean, median, maximum):
            raise ValueError(f'{lang}: 기준 chunk 통계 불일치: {actual}')
    return stats

def summarize_statistics(stats, lang, split):
    tokens, words, unks = int(stats.num_tokens.sum()), int(stats.num_words.sum()), int(stats.num_unk.sum())
    return {'lang':lang.upper(), 'split':split, 'n_docs':len(stats),
            'pct_over_512':round(float(stats.exceeds_512.mean()*100), 2),
            'pct_over_512_content_only':round(float(stats.exceeds_512_content_only.mean()*100), 2),
            'pct_over_512_with_query':round(float(stats.exceeds_512_with_query.mean()*100), 2),
            'mean_chunks':round(float(stats.num_chunks.mean()), 2),
            'expected_test_chunks':EXPECTED_TEST_STATS[lang][0] if split == 'test' else np.nan,
            'median_chunks':float(stats.num_chunks.median()), 'max_chunks':int(stats.num_chunks.max()),
            'mean_tokens':round(float(stats.num_tokens.mean()), 1), 'mean_words':round(float(stats.num_words.mean()), 1),
            'corpus_fertility':round(tokens / words, 4) if words else np.nan,
            'doc_mean_fertility':round(float(stats.fertility.mean()), 4),
            'token_count':tokens, 'word_count':words, 'unk_count':unks,
            'unk_rate':unks / tokens if tokens else 0.0}

split_json_path = DIRS['splits'] / 'celex_splits.json'
if not split_json_path.exists():
    raise FileNotFoundError(f'notebook 01의 공통 CELEX JSON을 먼저 배치하세요: {split_json_path}')
SPLIT_INFO = json.loads(split_json_path.read_text(encoding='utf-8'))
validate_split_ids(SPLIT_INFO)

# 최초 manifest를 유지한 채 legacy embedding을 hash로 검증합니다.
manifest_path = WORK_DIR / 'cache_manifest.json'
SOURCE_MANIFEST = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}
print('Split 검증 완료:', {k:len(SPLIT_INFO[k]) for k in SPLIT_SIZES})


Split 검증 완료: {'main_train': 10000, 'adaptation_pool': 4604, 'validation': 5000, 'test': 5000}


In [3]:
# 3. FP32 기본값, fp16 사전 검증, 모든 chunk vector를 저장하지 않는 집계
tokenizer = None
RESOLVED_MODEL_REVISION = None
FP16_VALIDATION = None

def initialize_tokenizer():
    global tokenizer, RESOLVED_MODEL_REVISION, PREFIX_ARR, SEP_ARR, PAD_ID, UNK_ID
    if tokenizer is None:
        from transformers import AutoConfig, AutoTokenizer
        config = AutoConfig.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
        RESOLVED_MODEL_REVISION = getattr(config, '_commit_hash', None)
        if not RESOLVED_MODEL_REVISION:
            raise ValueError('모델의 실제 commit hash를 확인할 수 없습니다. MODEL_REVISION을 확인하세요.')
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=RESOLVED_MODEL_REVISION, use_fast=True)
        tokenizer.model_max_length = 10_000_000
        query_ids = tokenizer.encode('query: ', add_special_tokens=False)
        if len(query_ids) != 3 or tokenizer.num_special_tokens_to_add(pair=False) != 2:
            raise ValueError('query 3 tokens + special 2 tokens 가정이 달라졌습니다.')
        PREFIX_ARR = np.asarray([tokenizer.cls_token_id] + query_ids, dtype=np.int32)
        SEP_ARR = np.asarray([tokenizer.sep_token_id], dtype=np.int32)
        PAD_ID, UNK_ID = tokenizer.pad_token_id, tokenizer.unk_token_id
    return tokenizer

def batch_tokenize_and_chunk_documents(texts, word_counts, celex_ids, chunk_size=480,
                                      overlap=0, max_chunks=None, tok_batch_size=2000):
    if chunk_size != 480 or overlap != 0 or max_chunks is not None:
        raise ValueError('Core는 chunk=480, overlap=0, max_chunks=None으로 고정합니다.')
    initialize_tokenizer()
    chunks, owners, rows = [], [], []
    for start in range(0, len(texts), tok_batch_size):
        encoded = tokenizer(texts[start:start+tok_batch_size], add_special_tokens=False,
                            truncation=False, return_attention_mask=False)['input_ids']
        for local, raw_ids in enumerate(encoded):
            doc_idx = start + local
            ids = np.asarray(raw_ids, dtype=np.int32)
            count = max(1, (len(ids)+479)//480)
            for pos in range(0, max(1, len(ids)), 480):
                chunks.append(np.concatenate([PREFIX_ARR, ids[pos:pos+480], SEP_ARR]))
                owners.append(doc_idx)
            rows.append({'celex_id':celex_ids[doc_idx], 'num_words':int(word_counts[doc_idx]),
                         'num_tokens':len(ids), 'num_chunks':count, 'num_unk':int((ids == UNK_ID).sum())})
    return chunks, np.asarray(owners, dtype=np.int64), pd.DataFrame(rows)

@torch.inference_mode()
def encode_and_pool_documents_gpu_fast(model, flat_chunks, chunk_doc_indices, n_docs,
                                      batch_size=256, device=DEVICE, compute_p2_first_chunk=True,
                                      compute_p3_maxpool=False, use_fp16=False, _precision_probe=False):
    if use_fp16 and not str(device).startswith('cuda'):
        raise ValueError('fp16은 CUDA에서만 사용할 수 있습니다.')
    if use_fp16 and not _precision_probe and not (FP16_VALIDATION and FP16_VALIDATION.get('passed')):
        raise ValueError('100문서 fp32/fp16 cosine 검증을 먼저 통과해야 합니다.')
    n_chunks = len(flat_chunks)
    if len(chunk_doc_indices) != n_chunks or not n_docs:
        raise ValueError('청크/문서 개수 오류')
    first_positions = np.full(n_docs, -1, dtype=np.int64)
    for pos, owner in enumerate(chunk_doc_indices):
        if not 0 <= owner < n_docs:
            raise ValueError('문서 index 범위 오류')
        if first_positions[owner] < 0:
            first_positions[owner] = pos
    if (first_positions < 0).any():
        raise ValueError('chunk가 없는 문서')
    lengths = np.asarray([len(c) for c in flat_chunks])
    if lengths.max() > 512:
        raise ValueError('512-token 입력 한도 초과')
    order = np.argsort(lengths, kind='stable')
    width = model.config.hidden_size
    sums = torch.zeros((n_docs, width), dtype=torch.float32, device=device)
    counts = torch.zeros((n_docs, 1), dtype=torch.float32, device=device)
    first = torch.zeros_like(sums) if compute_p2_first_chunk else None
    maxima = torch.full_like(sums, -float('inf')) if compute_p3_maxpool else None
    for start in tqdm(range(0, n_chunks, batch_size), desc='Encoding chunks', leave=False):
        positions = order[start:start+batch_size]
        data = np.full((len(positions), int(lengths[positions].max())), PAD_ID, dtype=np.int64)
        mask = np.zeros_like(data)
        for row, pos in enumerate(positions):
            seq = flat_chunks[pos]
            data[row, :len(seq)], mask[row, :len(seq)] = seq, 1
        input_ids = torch.as_tensor(data, device=device)
        attention = torch.as_tensor(mask, device=device)
        with torch.autocast(device_type=str(device).split(':')[0], dtype=torch.float16, enabled=use_fp16):
            hidden = model(input_ids=input_ids, attention_mask=attention).last_hidden_state.float()
        weights = attention.unsqueeze(-1).float()
        vectors = (hidden * weights).sum(1) / weights.sum(1).clamp_min(1)
        owners_np = chunk_doc_indices[positions]
        owners = torch.as_tensor(owners_np, dtype=torch.long, device=device)
        sums.index_add_(0, owners, vectors)
        counts.index_add_(0, owners, torch.ones((len(owners),1), device=device))
        if first is not None:
            keep = np.flatnonzero(positions == first_positions[owners_np])
            if len(keep):
                ki = torch.as_tensor(keep, device=device)
                first[owners[ki]] = vectors[ki]
        if maxima is not None:
            maxima.scatter_reduce_(0, owners[:,None].expand_as(vectors), vectors, reduce='amax', include_self=True)
    if not bool((counts > 0).all()):
        raise ValueError('집계되지 않은 문서')
    def normalized(value):
        return None if value is None else F.normalize(value, p=2, dim=1).cpu().numpy().astype(np.float32)
    return normalized(sums/counts), normalized(first), normalized(maxima)

def validate_fp16_precision(model):
    if not str(DEVICE).startswith('cuda'):
        raise ValueError('fp16 검증에는 CUDA가 필요합니다.')
    ids = sorted(np.random.default_rng(SEED).choice(SPLIT_INFO['validation'], size=100, replace=False).tolist())
    texts, words = load_lang_split_texts_and_word_counts('en', 'validation', ids)
    chunks, owners, _ = batch_tokenize_and_chunk_documents(texts, words, ids)
    common = dict(model=model, flat_chunks=chunks, chunk_doc_indices=owners, n_docs=100,
                  batch_size=ENCODE_BATCH_SIZE, device=DEVICE, compute_p2_first_chunk=False,
                  compute_p3_maxpool=False, _precision_probe=True)
    fp32, _, _ = encode_and_pool_documents_gpu_fast(**common, use_fp16=False)
    fp16, _, _ = encode_and_pool_documents_gpu_fast(**common, use_fp16=True)
    cosine = np.sum(fp32.astype(np.float64)*fp16.astype(np.float64), axis=1)
    record = {'passed':bool(np.all(cosine >= 0.9999)), 'minimum_cosine':float(cosine.min()),
              'required_cosine':0.9999, 'celex_ids':ids, 'model_revision':RESOLVED_MODEL_REVISION,
              'batch_size':ENCODE_BATCH_SIZE}
    write_json_atomic(DIRS['meta'] / 'fp16_validation.json', record)
    if not record['passed']:
        raise ValueError('fp16 검증 실패: USE_FP16=False로 다시 실행하세요.')
    return record


In [4]:
# 4. 캐시 재사용 검증, 통계만 보정, 라벨 및 생성 설정 보존
_RAW_HASHES = {}

def raw_file_hash(path):
    key = str(Path(path).resolve())
    if key not in _RAW_HASHES:
        _RAW_HASHES[key] = file_sha256(path)
    return _RAW_HASHES[key]

def partition_paths(lang, split):
    stem = f'{lang}_{split}'
    return {'core':DIRS['emb_core']/f'{stem}.npy',
            'ids':DIRS['emb_core']/f'{stem}_celex.json',
            'stats':DIRS['stats']/f'{stem}_doc_stats.parquet',
            'first':DIRS['emb_p2_first']/f'{stem}_first_chunk.npy',
            'maxpool':DIRS['emb_p3']/f'{stem}_maxpool.npy',
            'labels':DIRS['labels']/f'{stem}_Y.npy',
            'meta':DIRS['meta']/f'{stem}.json'}

def require_known_hash(path, old_meta=None, field=None):
    rel = Path(path).relative_to(WORK_DIR).as_posix()
    # 완료된 분할 metadata가 최초 manifest보다 최신일 수 있습니다(중단 후 재개).
    declared = old_meta.get('file_sha256', {}).get(field) if old_meta is not None else None
    if not declared:
        declared = SOURCE_MANIFEST.get(rel, {}).get('sha256')
    if not declared:
        raise ValueError(f'기존 파일의 신뢰할 hash가 없습니다: {rel}. 원 manifest를 함께 배치하세요.')
    actual = file_sha256(path)
    if not actual.startswith(declared):
        raise ValueError(f'캐시 hash 불일치: {rel}')
    return actual

def runtime_versions():
    result = {'python':platform.python_version()}
    for name in ['torch','transformers','numpy','pandas','scikit-learn']:
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = None
    return result

def current_generation_provenance(batch_size):
    return {'origin':'generated_by_this_notebook', 'model_name':MODEL_NAME,
            'model_revision':RESOLVED_MODEL_REVISION, 'tokenizer_revision':RESOLVED_MODEL_REVISION,
            'prefix':'query: ', 'content_chunk_tokens':480, 'overlap':0, 'max_chunk_cap':None,
            'chunk_normalization':False, 'token_pooling':'attention_mask_mean',
            'document_pooling':'unweighted_chunk_mean', 'document_normalization':'L2',
            'storage_dtype':'float32', 'compute_precision':'fp16_autocast' if USE_FP16 else 'fp32',
            'batch_size':batch_size, 'sorted_by_chunk_length':True,
            'fp16_validation':FP16_VALIDATION if USE_FP16 else None, 'versions':runtime_versions()}

def legacy_provenance():
    # 현재 실행 환경을 과거 encoder 환경으로 기록하지 않습니다.
    return {'origin':'legacy_cache_reused', 'model_name':None, 'model_revision':None,
            'tokenizer_revision':None, 'compute_precision':None, 'batch_size':None,
            'versions':None, 'declared_source_model':MODEL_NAME,
            'historical_settings_status':'not_recorded; original arrays retained and hash/IDs/norm verified'}

def run_all_core_embeddings_fast(batch_size=ENCODE_BATCH_SIZE):
    global FP16_VALIDATION
    summary_rows, label_reference, partition_records = [], {}, []
    model = None
    for lang in LANGUAGES:
        for split, raw_split in RAW_SPLITS.items():
            ids = SPLIT_INFO[split]
            paths = partition_paths(lang, split)
            save_first = split in ['main_train','validation','test']
            old_meta = json.loads(paths['meta'].read_text(encoding='utf-8')) if paths['meta'].exists() else {}
            existing = paths['core'].exists()
            raw_path = DATA_DIR / f'{lang}_{raw_split}.parquet'
            raw = load_aligned_raw(lang, raw_split, ids)
            texts = raw.text.tolist()
            words = np.asarray([len(t.split()) for t in texts], dtype=np.int32)
            y = labels_to_multihot(raw.labels)
            if split in label_reference and not np.array_equal(label_reference[split], y):
                raise ValueError(f'{lang}/{split}: 8개 언어 label mapping/count가 다릅니다.')
            label_reference.setdefault(split, y)
            if existing:
                if not paths['ids'].exists() or not paths['stats'].exists():
                    raise ValueError(f'{lang}/{split}: 기존 캐시의 ID 또는 통계 파일이 없습니다.')
                saved_ids = json.loads(paths['ids'].read_text(encoding='utf-8'))
                if saved_ids != ids:
                    raise ValueError(f'{lang}/{split}: embedding CELEX 순서 불일치')
                validate_embedding(paths['core'], ids)
                require_known_hash(paths['core'], old_meta, 'core')
                require_known_hash(paths['stats'], old_meta, 'stats')
                provenance = old_meta.get('embedding_provenance', legacy_provenance())
                if RUN_MODE == 'generate':
                    initialize_tokenizer()
                    expected = current_generation_provenance(batch_size)
                    for key in ['origin','model_name','model_revision','prefix','content_chunk_tokens',
                                'overlap','max_chunk_cap','chunk_normalization','token_pooling',
                                'document_pooling','document_normalization','compute_precision','batch_size']:
                        if provenance.get(key) != expected[key]:
                            raise ValueError(f'{lang}/{split}: {key}가 다른 캐시입니다. 새 WORK_DIR을 사용하세요.')
                if old_meta.get('raw_parquet_sha256') and old_meta['raw_parquet_sha256'] != raw_file_hash(raw_path):
                    raise ValueError(f'{lang}/{split}: raw parquet가 변경되었습니다.')
                for key in ['first','maxpool']:
                    if paths[key].exists():
                        validate_embedding(paths[key], ids)
                        require_known_hash(paths[key], old_meta, key)
                if save_first and not paths['first'].exists():
                    raise ValueError(f'{lang}/{split}: first-chunk 누락. 원 캐시에서 복원하거나 새 generate 폴더에서 생성하세요.')
                stats = pd.read_parquet(paths['stats'])
                print(f'[Reuse embedding / repair stats] {lang}/{split}: {len(ids):,}')
            else:
                if RUN_MODE != 'generate':
                    raise FileNotFoundError(f'{paths["core"]}: repair_existing은 GPU 재인코딩을 하지 않습니다.')
                if any(paths[k].exists() for k in ['ids','stats','first','maxpool','meta']):
                    raise ValueError(f'{lang}/{split}: 불완전한 기존 캐시입니다. 빈 출력 폴더를 사용하세요.')
                initialize_tokenizer()
                if model is None:
                    from transformers import AutoModel
                    model = AutoModel.from_pretrained(MODEL_NAME, revision=RESOLVED_MODEL_REVISION).to(DEVICE).float()
                    model.eval()
                    for parameter in model.parameters():
                        parameter.requires_grad = False
                    if USE_FP16:
                        FP16_VALIDATION = validate_fp16_precision(model)
                chunks, owners, stats = batch_tokenize_and_chunk_documents(texts, words, ids)
                core, first, maxpool = encode_and_pool_documents_gpu_fast(
                    model, chunks, owners, len(ids), batch_size=batch_size, device=DEVICE,
                    compute_p2_first_chunk=save_first, compute_p3_maxpool=RUN_P3_MAXPOOL and save_first,
                    use_fp16=USE_FP16)
                save_npy_atomic(paths['core'], core)
                write_json_atomic(paths['ids'], ids)
                if first is not None:
                    save_npy_atomic(paths['first'], first)
                if maxpool is not None:
                    save_npy_atomic(paths['maxpool'], maxpool)
                provenance = current_generation_provenance(batch_size)
                del chunks, owners, core, first, maxpool
            stats = repair_document_statistics(stats, texts, ids, lang, split)
            backup_before_repair(paths['stats'])
            tmp = paths['stats'].with_suffix('.parquet.tmp')
            stats.to_parquet(tmp, index=False)
            os.replace(tmp, paths['stats'])
            save_npy_atomic(paths['labels'], y)
            write_json_atomic(DIRS['labels']/f'{lang}_{split}_celex.json', ids)
            record = {'schema_version':CODE_SCHEMA_VERSION, 'language':lang, 'split':split,
                      'n_documents':len(ids), 'label_column_order':LABEL_INDICES,
                      'celex_order':'sorted canonical 01_splits/celex_splits.json',
                      'raw_parquet_path':str(raw_path), 'raw_parquet_sha256':raw_file_hash(raw_path),
                      'embedding_provenance':provenance,
                      'statistics_definition':{'words':'len(raw_text.split())', 'tokens':'body only; no query/special tokens',
                                               'fertility':'body tokens / Python whitespace words',
                                               'exceeds_512':'body tokens + CLS/SEP(2) > 512',
                                               'exceeds_512_with_query':'body tokens + query(3) + special(2) > 512'},
                      'statistics_repair_versions':runtime_versions(),
                      'file_sha256':{key:file_sha256(paths[key]) for key in ['core','ids','stats','labels','first','maxpool'] if paths[key].exists()}}
            backup_before_repair(paths['meta'])
            write_json_atomic(paths['meta'], record)
            partition_records.append({'language':lang, 'split':split, 'n_documents':len(ids),
                                      'origin':provenance['origin'], 'first_chunk':paths['first'].exists()})
            summary_rows.append(summarize_statistics(stats, lang, split))
            del raw, texts, words, stats, y
    summary = pd.DataFrame(summary_rows)
    summary_path = DIRS['stats']/'tokenization_and_chunk_summary_8langs.csv'
    backup_before_repair(summary_path)
    summary.to_csv(summary_path, index=False)
    write_json_atomic(DIRS['meta']/'label_mapping.json', {
        'dataset':'MultiEURLEX', 'level':1, 'column_order':LABEL_INDICES,
        'meaning':'Y[:,j] corresponds to the original parquet labels index j; no remapping',
        'label_names':None, 'label_names_note':'Semantic names were not included in supplied parquet metadata.'})
    write_json_atomic(DIRS['meta']/'cache_metadata.json', {
        'schema_version':CODE_SCHEMA_VERSION, 'run_mode':RUN_MODE, 'language_order':LANGUAGES,
        'split_sizes':SPLIT_SIZES, 'partitions':partition_records,
        'source_notebook':'Embedding_2020503660.ipynb', 'repair_backup':str(REPAIR_BACKUP_DIR),
        'historical_unknowns_preserved':True})
    return summary


In [5]:
# 4-1. 기본 캐시 보정/생성 실행 및 standalone loader/설명서 저장
CACHE_LOADER_SOURCE = '"""Load aligned E5 embeddings, Level-1 multi-hot labels and CELEX IDs."""\nfrom pathlib import Path\nimport json\nimport hashlib\nimport numpy as np\n\ndef _sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda:f.read(1024*1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef load_cache(root, lang, split, variant=\'core\', verify_hash=True, mmap_mode=None):\n    root = Path(root)\n    lang = lang.lower()\n    if split not in [\'main_train\',\'adaptation_pool\',\'validation\',\'test\']:\n        raise ValueError(\'Unknown split\')\n    ids = json.loads((root/\'01_splits/celex_splits.json\').read_text(encoding=\'utf-8\'))[split]\n    stem = f\'{lang}_{split}\'\n    meta = json.loads((root/f\'cache_metadata/{stem}.json\').read_text(encoding=\'utf-8\'))\n    if variant == \'core\':\n        x_path, field = root/f\'02_embeddings_core_480_mean/{stem}.npy\', \'core\'\n    elif variant == \'first_chunk\':\n        x_path, field = root/f\'02_embeddings_p2_first_chunk/{stem}_first_chunk.npy\', \'first\'\n    else:\n        raise ValueError(\'variant must be core or first_chunk\')\n    y_path = root/f\'03_labels/{stem}_Y.npy\'\n    if meta[\'label_column_order\'] != list(range(21)):\n        raise ValueError(\'Unexpected label column order\')\n    core_ids = json.loads((root/f\'02_embeddings_core_480_mean/{stem}_celex.json\').read_text(encoding=\'utf-8\'))\n    label_ids = json.loads((root/f\'03_labels/{stem}_celex.json\').read_text(encoding=\'utf-8\'))\n    if core_ids != ids or label_ids != ids:\n        raise ValueError(\'Embedding/label CELEX order mismatch\')\n    if verify_hash:\n        for path, key in [(x_path, field), (y_path, \'labels\')]:\n            expected = meta[\'file_sha256\'].get(key)\n            if not expected or _sha256(path) != expected:\n                raise ValueError(f\'Hash mismatch: {path}\')\n    x = np.load(x_path, allow_pickle=False, mmap_mode=mmap_mode)\n    y = np.load(y_path, allow_pickle=False, mmap_mode=mmap_mode)\n    if x.shape != (len(ids),384) or y.shape != (len(ids),21):\n        raise ValueError(\'Invalid embedding/label shape\')\n    if not np.isin(y, [0,1]).all():\n        raise ValueError(\'Labels must be binary multi-hot\')\n    return x, y, ids\n'
CACHE_README = "# Cache 사용 설명\n\n원본 split JSON과 모든 배열은 같은 CELEX 정렬 순서입니다. Y 열은 원본 Level 1 index 0~20입니다.\n\n```python\nfrom cache_loader import load_cache\nX, Y, ids = load_cache('/content/drive/MyDrive/기계학습_산출물', 'en', 'test')\nX_first, Y, ids = load_cache('/content/drive/MyDrive/기계학습_산출물', 'en', 'test', variant='first_chunk')\n```\n\n- `02_embeddings_core_480_mean`: 8×4 core 배열, 384차원 float32.\n- `02_embeddings_p2_first_chunk`: main/validation/test의 first-chunk 배열.\n- `03_labels`: document×21 multi-hot와 CELEX 목록.\n- `02_tokenization_stats`: Python whitespace word count, 본문 tokens/chunks/UNK, corpus 및 document-average fertility.\n- `cache_metadata`: 분할별 hash, 실제 생성 시 model/tokenizer commit·fp32/fp16·batch·환경, 원문 hash, 숫자 label 순서.\n- `08_first_chunk_robustness`: 실행한 경우에만 core/first-chunk LR 비교표·test 확률·CELEX·확정 설정.\n\n기존 cache는 역사적 생성 환경이 기록되지 않았으므로 `legacy_cache_reused`로 표시합니다. 현재 보정 환경을 과거 생성 환경으로 대체하지 않습니다. stats/summary와 RQ4의 word/fertility 열은 보정 실행 후 갱신됩니다. core embedding은 바뀌지 않습니다.\n\n기존 manifest가 없는 legacy cache는 자동 신뢰하지 않습니다. 원 산출물의 cache_manifest.json을 함께 배치하세요. 각 분할의 model/config/ID/hash가 다르면 자동 덮어쓰기하지 않고 오류를 냅니다.\n\nfirst-chunk 실험은 B의 EN Standard 및 언어별 Direct의 확정 C/threshold를 core와 first-chunk에 동일 적용합니다. 결과를 보고 설정을 바꾸지 않습니다. label별 threshold, StandardScaler, class weighting은 사용하지 않습니다.\n"

exec(CACHE_LOADER_SOURCE, globals())
df_tok_summary = run_all_core_embeddings_fast(batch_size=ENCODE_BATCH_SIZE)
for filename, content in [('cache_loader.py',CACHE_LOADER_SOURCE), ('CACHE_README.md',CACHE_README)]:
    target = WORK_DIR/filename
    backup_before_repair(target)
    target.write_text(content,encoding='utf-8')
display(df_tok_summary[df_tok_summary.split == 'test'])


[Reuse embedding / repair stats] en/main_train: 10,000
[Reuse embedding / repair stats] en/adaptation_pool: 4,604
[Reuse embedding / repair stats] en/validation: 5,000
[Reuse embedding / repair stats] en/test: 5,000
[Reuse embedding / repair stats] de/main_train: 10,000
[Reuse embedding / repair stats] de/adaptation_pool: 4,604
[Reuse embedding / repair stats] de/validation: 5,000
[Reuse embedding / repair stats] de/test: 5,000
[Reuse embedding / repair stats] fr/main_train: 10,000
[Reuse embedding / repair stats] fr/adaptation_pool: 4,604
[Reuse embedding / repair stats] fr/validation: 5,000
[Reuse embedding / repair stats] fr/test: 5,000
[Reuse embedding / repair stats] el/main_train: 10,000
[Reuse embedding / repair stats] el/adaptation_pool: 4,604
[Reuse embedding / repair stats] el/validation: 5,000
[Reuse embedding / repair stats] el/test: 5,000
[Reuse embedding / repair stats] fi/main_train: 10,000
[Reuse embedding / repair stats] fi/adaptation_pool: 4,604
[Reuse embedding / rep

,lang,split,n_docs,pct_over_512,pct_over_512_content_only,pct_over_512_with_query,mean_chunks,expected_test_chunks,median_chunks,max_chunks,mean_tokens,mean_words,corpus_fertility,doc_mean_fertility,token_count,word_count,unk_count,unk_rate
3,EN,test,5000,67.58,67.48,67.78,5.75,5.75,2.0,574,2549.1,1845.9,1.3810,1.4295,12745693,9229596,0,0.0
7,DE,test,5000,72.02,71.96,72.16,6.42,6.42,2.0,630,2857.7,1691.4,1.6896,1.7000,14288333,8456834,0,0.0
11,FR,test,5000,74.92,74.78,75.24,7.02,7.02,2.0,715,3140.8,2012.5,1.5606,1.5767,15703905,10062515,0,0.0
15,EL,test,5000,91.52,91.48,91.60,7.81,7.81,3.0,807,3500.6,1911.2,1.8317,1.8994,17503023,9555781,0,0.0
19,FI,test,5000,75.56,75.34,75.78,6.64,6.64,2.0,679,2932.4,1308.1,2.2418,2.2230,14661986,6540341,0,0.0
23,PL,test,5000,78.06,77.78,78.56,6.75,6.75,2.0,694,2980.6,1643.0,1.8141,1.8564,14902797,8214790,0,0.0
27,BG,test,5000,73.74,73.28,74.10,6.48,6.48,2.0,648,2891.6,1824.3,1.5851,1.6000,14458041,9121492,0,0.0
31,MT,test,5000,98.18,98.18,98.24,10.91,10.91,3.0,1142,5007.4,1522.9,3.2880,3.3190,25036822,7614719,0,0.0


In [6]:
# 5. 전처리는 GPU/CPU에서 동일. TF-IDF 재계산은 설정으로 선택합니다.
DIGIT_REGEX = re.compile(r'\d')
WHITESPACE_REGEX = re.compile(r'\s+')

def preprocess(text):
    return WHITESPACE_REGEX.sub(' ', DIGIT_REGEX.sub('0', unicodedata.normalize('NFKC', text).lower())).strip()

def batch_preprocess_texts_tfidf(texts):
    return [preprocess(text) for text in texts]

def run_and_cache_all_tfidf():
    import joblib
    rows = []
    for lang in LANGUAGES:
        clean = {}
        for split in ['main_train','validation','test']:
            texts, _ = load_lang_split_texts_and_word_counts(lang, RAW_SPLITS[split], SPLIT_INFO[split])
            clean[split] = batch_preprocess_texts_tfidf(texts)
        for tag, ngram in [('unigram',(1,1)), ('bigram',(1,2))]:
            paths = {s:DIRS['tfidf_rq1']/f'{lang}_{s}_{tag}.npz' for s in clean}
            vec_path = DIRS['tfidf_rq1']/f'{lang}_{tag}_vectorizer.joblib'
            config_path = DIRS['tfidf_rq1']/f'{lang}_{tag}_config.json'
            config = {'ngram_range':list(ngram), 'min_df':5, 'max_df':0.5, 'sublinear_tf':True,
                      'preprocessing':'NFKC -> lower -> each digit to 0 -> Python whitespace',
                      'train_celex_sha256':hashlib.sha256(json.dumps(SPLIT_INFO['main_train']).encode()).hexdigest(),
                      'sklearn':version('scikit-learn')}
            if all(p.exists() for p in paths.values()) and vec_path.exists() and config_path.exists():
                if json.loads(config_path.read_text(encoding='utf-8')) != config:
                    raise ValueError(f'{lang}/{tag}: TF-IDF 설정 불일치. 새 출력 폴더를 사용하세요.')
                X_train = sp.load_npz(paths['main_train'])
            else:
                if any(p.exists() for p in paths.values()):
                    raise ValueError(f'{lang}/{tag}: 기존 행렬을 보존합니다. TF-IDF를 다시 만들려면 빈 tfidf 폴더를 사용하세요.')
                vec = SklearnTfidfVectorizer(ngram_range=ngram, min_df=5, max_df=0.5, sublinear_tf=True,
                                             lowercase=False, stop_words=None, dtype=np.float32)
                X_train = vec.fit_transform(clean['main_train'])
                sp.save_npz(paths['main_train'], X_train)
                for split in ['validation','test']:
                    sp.save_npz(paths[split], vec.transform(clean[split]))
                joblib.dump(vec, vec_path)
                write_json_atomic(config_path, config)
            rows.append({'lang':lang.upper(), 'ngram':tag, 'vocab_size':X_train.shape[1],
                         'train_shape':str(X_train.shape)})
    frame = pd.DataFrame(rows)
    frame.to_csv(DIRS['tfidf_rq1']/'tfidf_vocab_summary.csv', index=False)
    return frame

if RUN_TFIDF:
    display(run_and_cache_all_tfidf())
else:
    print('TF-IDF 함수 준비 완료. 기존 행렬은 보존합니다. 새 생성 시 RUN_TFIDF=True로 설정하세요.')


TF-IDF 함수 준비 완료. 기존 행렬은 보존합니다. 새 생성 시 RUN_TFIDF=True로 설정하세요.


In [7]:
# 6. 같은 score 행렬에서 retrieval과 cosine을 계산하고 수정된 fertility를 사용합니다.
def compute_rq4_alignment_rapids(include_p3_mrr=RUN_P3_MRR):
    # 함수 이름은 기존 호출과 호환. CPU float64로 backend와 동점 규칙을 고정합니다.
    en, _, en_ids = load_cache(WORK_DIR, 'en', 'test')
    en = np.asarray(en, dtype=np.float64)
    en /= np.linalg.norm(en, axis=1, keepdims=True)
    tok = pd.read_csv(DIRS['stats']/'tokenization_and_chunk_summary_8langs.csv')
    tok = tok[tok.split == 'test'].set_index('lang')
    rows = []
    for lang in TARGET_LANGUAGES:
        target, _, target_ids = load_cache(WORK_DIR, lang, 'test')
        if target_ids != en_ids:
            raise ValueError('Alignment의 EN/target CELEX 순서가 다릅니다.')
        target = np.asarray(target, dtype=np.float64)
        target /= np.linalg.norm(target, axis=1, keepdims=True)
        hits, diagonal, reciprocals = 0, [], []
        for start in range(0, len(target), 256):
            scores = target[start:start+256] @ en.T
            gold = np.arange(start, start+len(scores))
            same = scores[np.arange(len(scores)), gold]
            hits += int((scores.argmax(axis=1) == gold).sum())
            diagonal.extend(same.tolist())
            if include_p3_mrr:
                reciprocals.extend((1.0 / ((scores > same[:,None]).sum(axis=1)+1)).tolist())
        row = {'lang':lang.upper(), 'top1_retrieval_acc':round(hits/len(target),4),
               'same_celex_cosine_mean':round(float(np.mean(diagonal)),4),
               'same_celex_cosine_sd':round(float(np.std(diagonal)),4),
               'p3_alignment_mrr':round(float(np.mean(reciprocals)),4) if include_p3_mrr else np.nan}
        for key in ['mean_chunks','mean_tokens','mean_words','corpus_fertility','doc_mean_fertility']:
            row[key] = tok.loc[lang.upper(),key]
        rows.append(row)
    result = pd.DataFrame(rows)
    output = DIRS['rq4']/'rq4_alignment_and_fertility_table.csv'
    backup_before_repair(output)
    result.to_csv(output, index=False)
    write_json_atomic(DIRS['meta']/'alignment_config.json', {
        'backend':'numpy CPU float64', 'score':'cosine after explicit L2 normalization',
        'query':'target test', 'candidates':'EN test', 'top1_tie_break':'first CELEX in sorted order',
        'mrr_ties':'optimistic rank; strictly greater scores + 1', 'mrr_enabled':include_p3_mrr,
        'fertility_source':'repaired body tokens / len(text.split())',
        'numpy_version':np.__version__})
    return result

if RUN_ALIGNMENT:
    display(compute_rq4_alignment_rapids())
else:
    print('Alignment 재계산 생략. RUN_ALIGNMENT=True로 설정하면 수정 통계로 다시 작성합니다.')


,lang,top1_retrieval_acc,same_celex_cosine_mean,same_celex_cosine_sd,p3_alignment_mrr,mean_chunks,mean_tokens,mean_words,corpus_fertility,doc_mean_fertility
0,DE,0.7440,0.9254,0.0127,NaN,6.42,2857.7,1691.4,1.6896,1.7000
1,FR,0.7588,0.9451,0.0106,NaN,7.02,3140.8,2012.5,1.5606,1.5767
2,EL,0.4144,0.9157,0.0151,NaN,7.81,3500.6,1911.2,1.8317,1.8994
3,FI,0.4492,0.9062,0.0148,NaN,6.64,2932.4,1308.1,2.2418,2.2230
4,PL,0.6002,0.9206,0.0140,NaN,6.75,2980.6,1643.0,1.8141,1.8564
5,BG,0.6608,0.9211,0.0129,NaN,6.48,2891.6,1824.3,1.5851,1.6000
6,MT,0.5276,0.9358,0.0106,NaN,10.91,5007.4,1522.9,3.2880,3.3190


In [8]:
# 7. 고정된 lbfgs 공통 분류기와 first-chunk robustness 실행 코드
def fit_predict_ovr_logreg_cuml(X_train, Y_train, X_eval, C=10.0, max_iter=2000):
    """기존 함수명 호환. 설계서의 lbfgs를 지키기 위해 sklearn CPU를 사용합니다."""
    from sklearn.linear_model import LogisticRegression
    if Y_train.shape != (len(X_train),21):
        raise ValueError('Y_train은 document × 21 label이어야 합니다.')
    probabilities = np.zeros((len(X_eval),21), dtype=np.float32)
    for label in range(21):
        y = Y_train[:,label]
        if np.unique(y).size < 2:
            probabilities[:,label] = float(y.mean())
        else:
            classifier = LogisticRegression(C=float(C), solver='lbfgs', max_iter=max_iter,
                                            class_weight=None, random_state=SEED)
            classifier.fit(X_train, y)
            probabilities[:,label] = classifier.predict_proba(X_eval)[:,1]
    return probabilities

def multilabel_metrics(y_true, probabilities, threshold):
    from sklearn.metrics import f1_score
    if y_true.shape != probabilities.shape or y_true.shape[1] != 21:
        raise ValueError('정답/확률 shape 불일치')
    predictions = probabilities >= threshold
    relevant = y_true.sum(axis=1).astype(int)
    order = np.argsort(-probabilities, axis=1, kind='stable')
    ranked_truth = np.take_along_axis(y_true, order, axis=1)
    top_r = np.arange(21)[None,:] < relevant[:,None]
    rp = np.divide((ranked_truth*top_r).sum(axis=1), relevant,
                   out=np.zeros(len(y_true), dtype=float), where=relevant > 0)
    return {'micro_f1':float(f1_score(y_true,predictions,average='micro',zero_division=0)),
            'macro_f1':float(f1_score(y_true,predictions,average='macro',zero_division=0)),
            'mrp':float(rp.mean())}

def validate_classifier_settings(settings, languages=LANGUAGES):
    if settings.get('status') != 'frozen_after_core_results':
        raise ValueError('B의 핵심 실험 확정값을 입력하고 status를 frozen_after_core_results로 설정하세요.')
    for lang in languages:
        row = settings.get('languages',{}).get(lang,{})
        if row.get('C') not in [1,10,100,1000]:
            raise ValueError(f'{lang}: 확정 C가 필요합니다.')
        threshold = row.get('threshold')
        if not isinstance(threshold,(int,float)) or not 0.05 <= threshold <= 0.50:
            raise ValueError(f'{lang}: 확정 threshold(0.05~0.50)가 필요합니다.')
    return settings['languages']

def run_first_chunk_robustness(settings_path, core_results_finalized=False):
    if not core_results_finalized:
        raise ValueError('핵심 실험 확정 뒤 CORE_RESULTS_FINALIZED=True로 설정하세요.')
    settings = json.loads(Path(settings_path).read_text(encoding='utf-8'))
    fixed = validate_classifier_settings(settings)
    rows = []
    output_dir = DIRS['robustness']
    for variant in ['core','first_chunk']:
        x_en, y_en, _ = load_cache(WORK_DIR,'en','main_train',variant)
        test_parts = [load_cache(WORK_DIR,lang,'test',variant) for lang in LANGUAGES]
        all_test = np.concatenate([part[0] for part in test_parts])
        english_probabilities = fit_predict_ovr_logreg_cuml(x_en,y_en,all_test,C=fixed['en']['C'])
        offset = 0
        for lang, (x_test,y_test,ids) in zip(LANGUAGES,test_parts):
            standard = english_probabilities[offset:offset+len(ids)]
            offset += len(ids)
            if lang == 'en':
                direct = standard
            else:
                x_train,y_train,_ = load_cache(WORK_DIR,lang,'main_train',variant)
                direct = fit_predict_ovr_logreg_cuml(x_train,y_train,x_test,C=fixed[lang]['C'])
            standard_metrics = multilabel_metrics(y_test,standard,fixed['en']['threshold'])
            direct_metrics = multilabel_metrics(y_test,direct,fixed[lang]['threshold'])
            record = {'variant':variant,'lang':lang.upper(),'standard_C':fixed['en']['C'],
                      'standard_threshold':fixed['en']['threshold'],'direct_C':fixed[lang]['C'],
                      'direct_threshold':fixed[lang]['threshold']}
            for key,value in standard_metrics.items(): record['standard_'+key] = value
            for key,value in direct_metrics.items(): record['direct_'+key] = value
            record['transfer_gap'] = direct_metrics['micro_f1']-standard_metrics['micro_f1']
            rows.append(record)
            for condition,probs in [('standard',standard),('direct',direct)]:
                save_npy_atomic(output_dir/f'{variant}_{lang}_{condition}_test_probs.npy',probs)
            write_json_atomic(output_dir/f'{variant}_{lang}_test_celex.json',ids)
    result = pd.DataFrame(rows)
    result.to_csv(output_dir/'first_chunk_robustness.csv',index=False)
    metrics = ['standard_micro_f1','standard_macro_f1','standard_mrp',
               'direct_micro_f1','direct_macro_f1','direct_mrp','transfer_gap']
    core = result[result.variant=='core'].set_index('lang')[metrics]
    first = result[result.variant=='first_chunk'].set_index('lang')[metrics]
    delta = first-core
    delta.columns = [name+'_first_minus_core' for name in delta.columns]
    delta.to_csv(output_dir/'first_chunk_minus_core.csv')
    write_json_atomic(output_dir/'experiment_metadata.json',{
        'settings':settings, 'settings_sha256':file_sha256(settings_path),
        'solver':'sklearn lbfgs','max_iter':2000,'class_weight':None,'scaler':None,
        'comparison':'retrain LR only; frozen B C and thresholds shared by core/first-chunk',
        'test_used_for_selection':False,'versions':runtime_versions()})
    return result,delta

template_path = DIRS['meta']/'classifier_settings.template.json'
if not template_path.exists():
    write_json_atomic(template_path,{'status':'fill_from_B_then_freeze',
        'languages':{lang:{'C':None,'threshold':None} for lang in LANGUAGES},
        'note':'EN values are English-validation Standard settings; target values are target-validation Direct settings. Do not select using test scores.'})
if RUN_FIRST_CHUNK_ROBUSTNESS:
    first_chunk_results, first_chunk_delta = run_first_chunk_robustness(
        CLASSIFIER_SETTINGS_PATH, core_results_finalized=CORE_RESULTS_FINALIZED)
    display(first_chunk_results)
    display(first_chunk_delta)
else:
    print('first-chunk LR 코드 준비 완료. Gate 3 이후 B 설정 JSON과 두 실행 플래그를 지정하세요.')


first-chunk LR 코드 준비 완료. Gate 3 이후 B 설정 JSON과 두 실행 플래그를 지정하세요.


In [9]:
# 8. 전체 행/분할/라벨 검증과 manifest 작성
def verify_and_package_all_outputs(create_zip=CREATE_ZIP):
    validate_split_ids(SPLIT_INFO)
    total = 0
    for lang in LANGUAGES:
        for split,size in SPLIT_SIZES.items():
            x,y,ids = load_cache(WORK_DIR,lang,split,mmap_mode='r')
            validate_embedding(partition_paths(lang,split)['core'],ids)
            stats = pd.read_parquet(partition_paths(lang,split)['stats'])
            if len(stats) != size or stats.celex_id.tolist() != ids:
                raise ValueError(f'{lang}/{split}: 통계 행 불일치')
            if not np.allclose(stats.fertility, stats.num_tokens/stats.num_words, equal_nan=True):
                raise ValueError('Fertility mismatch')
            if not np.array_equal(stats.exceeds_512,(stats.num_tokens+2)>512):
                raise ValueError('512-token diagnostic mismatch')
            if not np.array_equal(stats.num_chunks,np.maximum(1,(stats.num_tokens+479)//480)):
                raise ValueError('Chunk count mismatch')
            if split in ['main_train','validation','test']:
                load_cache(WORK_DIR,lang,split,variant='first_chunk',mmap_mode='r')
                validate_embedding(partition_paths(lang,split)['first'],ids)
            total += len(ids)
    if total != 196832:
        raise ValueError(f'Expected 196832 documents, found {total}')
    manifest = {}
    for folder in DIRS.values():
        for path in sorted(folder.glob('*')):
            if not path.is_file() or path.name.endswith('.tmp'):
                continue
            info = {'size_mb':round(path.stat().st_size/(1024*1024),2),'sha256':file_sha256(path)}
            if path.suffix == '.npy':
                array = np.load(path,mmap_mode='r',allow_pickle=False)
                info.update(shape=list(array.shape),dtype=str(array.dtype))
            manifest[path.relative_to(WORK_DIR).as_posix()] = info
    for name in ['cache_loader.py','CACHE_README.md']:
        path = WORK_DIR/name
        manifest[name] = {'size_mb':round(path.stat().st_size/(1024*1024),2),'sha256':file_sha256(path)}
    backup_before_repair(WORK_DIR/'cache_manifest.json')
    write_json_atomic(WORK_DIR/'cache_manifest.json',manifest)
    print(f'검증 완료: 32 partitions / {total:,} documents / 21 label columns')
    if create_zip:
        import zipfile
        # 기존 압축 파일을 덮어쓰지 않고 현재 manifest의 파일만 압축합니다.
        zip_path = WORK_DIR.parent/f'multi_eurlex_cache_{REPAIR_RUN_ID}.zip'
        with zipfile.ZipFile(zip_path,'x',compression=zipfile.ZIP_DEFLATED) as archive:
            for rel in list(manifest)+['cache_manifest.json']:
                archive.write(WORK_DIR/rel,arcname=rel)
        print(f'Created: {zip_path}')
    return manifest

manifest_result = verify_and_package_all_outputs()


검증 완료: 32 partitions / 196,832 documents / 21 label columns
